In [26]:
import subprocess
import sys
import random as rd
import webbrowser

def instalar(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import pandas as pd
except ImportError:
    instalar('pandas')
    import pandas as pd

In [27]:
#Criar backup
try:
    backup = pd.read_csv('save.csv')
except:
    backup = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
backup.to_csv('save_backup.csv', index=False)

In [28]:
def escolhe_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    for index, row in data.iterrows(): 
        print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]}')

In [29]:
def ver_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
    except:
        print('Você ainda não tem nenhuma carta na coleção! Abra alguns pacotes para começar a colecionar!')

In [30]:
def rodar_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    pacote = []
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity'])

    commons = data[data['rarity'] == 'C']
    c_tiradas = commons.sample(n=4).to_dict('records')
    pacote.extend(c_tiradas)

    luck = rd.randint(1, 100)

    if luck <= 70:
        raridade = 'R'
    elif luck <= 90:
        raridade = 'RR'
    elif luck <= 99:
        raridade = 'RRR'
    else:
        raridade = 'SP'

    raras = data[data['rarity'] == raridade]
    r_tirada = raras.sample(n=1).to_dict('records')
    pacote.extend(r_tirada)

    pacote = pd.DataFrame(pacote)
    
    return pacote

In [31]:
def atualizar_save(box, pacote):
    #Quero colocar os valores do pacote na box e se houver repetidos, qtt += 1
    
    for index, row in pacote.iterrows():
        card = box[(box['set'] == row['set']) & (box['id'] == row['id'])]
        if not card.empty:
            box.loc[card.index, 'qtt'] += 1
        else:
            new_card = row.to_dict()
            new_card['qtt'] = 1
            #box não tem append, então vou criar um novo dataframe com a nova linha e concatenar com a box
            new_card_df = pd.DataFrame([new_card])
            box = pd.concat([box, new_card_df], ignore_index=True)

    return box

In [32]:
def rodar_box(name, qtt):
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    
    for i in range(qtt):
        pacote = rodar_pacote(name)
        box = atualizar_save(box, pacote)
    
    try:
        save = pd.read_csv('save.csv')
    except:
        save = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])

    box.sort_values(by=['id'], inplace=True)
    box.to_csv('last_box.csv', index=False)

    save = atualizar_save(save, box)
    save.sort_values(by=['set', 'id'], inplace=True)
    save.to_csv('save.csv', index=False)
    
    return box

In [33]:
def gerar_link_wiki(nome_carta):
    # A wiki usa underscores no lugar de espaços
    nome_formatado = nome_carta.replace(' ', '_')
    return f'https://cardfight.fandom.com/wiki/{nome_formatado}'

In [34]:
def web_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            link = gerar_link_wiki(row['name'])
            print(f'Abrindo Wiki de: {row["name"]}...')
            webbrowser.open(link) # Isso abre o navegador automaticamente
    except:
        print('Erro ao carregar coleção.')

In [99]:
pip install requests beautifulsoup4 pandas

     ---------------------------------------- 64.9/64.9 kB 3.4 MB/s eta 0:00:00
     -------------------------------------- 107.7/107.7 kB 3.1 MB/s eta 0:00:00
     -------------------------------------- 153.9/153.9 kB 4.6 MB/s eta 0:00:00
  Using cached idna-3.11-py3-none-any.whl (71 kB)
     -------------------------------------- 131.6/131.6 kB 3.9 MB/s eta 0:00:00
     -------------------------------------- 153.7/153.7 kB 3.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [102]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv

def scrape_vanguard_v1(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    print(f"Lendo dados de: {url}...")
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Erro ao acessar a URL: {e}")
        return

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Nome do set extraído do título da página
    set_name = soup.find('h1', id='firstHeading').text.strip()
    
    cards_list = []
    
    # Procuramos todas as tabelas com a classe 'wikitable'
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    for table in tables:
        rows = table.find_all('tr')
        # Verifica se a tabela tem cabeçalho (pula se não tiver)
        if not rows:
            continue
            
        for row in rows:
            cols = row.find_all(['td', 'th'])
            
            # Filtro para ignorar linhas de cabeçalho (que contêm "Name", "Grade", etc.)
            if len(cols) < 6 or "Name" in cols[1].text:
                continue
                
            try:
                card_info = {
                    'set': set_name,
                    'id': cols[0].text.strip(),
                    'name': cols[1].text.strip(),
                    'grade': cols[2].text.strip(),
                    'clan': cols[3].text.strip(),
                    'type': cols[4].text.strip(),
                    'rarity': cols[5].text.strip()
                }
                cards_list.append(card_info)
            except IndexError:
                continue

    if cards_list:
        df = pd.DataFrame(cards_list)
        # Limpeza simples para remover notas como "[1]" que às vezes aparecem no texto
        df = df.replace(r'\[.*\]', '', regex=True)
        
        filename = "vanguard_cards.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)
        print(f"--- Processo Concluído ---")
        print(f"Total de cartas encontradas: {len(cards_list)}")
        print(f"Arquivo salvo como: {filename}")
    else:
        print("Nenhuma carta foi encontrada. Verifique se o link está correto.")

# Execução direta com o link de exemplo
link = "https://cardfight.fandom.com/wiki/Booster_Set_3:_Demonic_Lord_Invasion"
scrape_vanguard_v1(link)

Lendo dados de: https://cardfight.fandom.com/wiki/Booster_Set_3:_Demonic_Lord_Invasion...
Erro ao acessar a URL: 403 Client Error: Forbidden for url: https://cardfight.fandom.com/wiki/Booster_Set_3:_Demonic_Lord_Invasion
